# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion

In [6]:
# define file paths
analysis_base_path = "/accounts/projects/binyu/hao_huang/stat-genie/examples/output/"
analysis_subdir_path = join(analysis_base_path, "fertility", "shuffle_true")
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"
# create llm assistant
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-11 03:23:36.47][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


## Extract Features **X** Used in Model

In [3]:
# create dict to store features
# features = {}

In [4]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # create internal dict for analysis features
#     features[i] = {}
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
#     control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in ind_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     for dict_idx, var in enumerate(ind_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
        
#     # save updated independent variables in features dict
#     features[i]['independent_variables'] = ind_vars
    
#     # tkae same approach for control variables
#     for dict_idx, var in enumerate(control_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
    
#     # save updated control variables in features dict
#     features[i]['control_variables'] = control_vars

In [5]:
# view feature dictionary to ensure correctness
# features

## Extract Response *y* used in Model

In [6]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in response_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     # for dict_idx, var in enumerate(response_vars):
#     transform_responses = get_feature_transforms(llm_assistant,
#                                                  transform_code,
#                                                  response_vars['columns'],
#                                                  response_vars['description'])
#     response_vars['transform_code'] = [response.text[0].content \
#         for response in transform_responses]

#     # save updated response variables in features dict
#     features[i]['response_variables'] = response_vars

In [7]:
# view feature dictionary to ensure correctness
# features

## Extract Features **X** and *y* Used in Model

In [8]:
features = format_features(multirun_analyses, num_analyses, llm_assistant)

In [9]:
features

{0: {'independent_variables': [{'description': 'Binary indicator for whether the player has a dark skin tone based on photo raters (1 = dark skin, 0 = light skin). Constructed by averaging the two independent rater scores (rater1, rater2) which are normalized to 0-1 and then selecting extreme ratings: dark if average >= 0.6, light if average <= 0.4 (middle/ambiguous cases are excluded).',
    'columns': ['DarkSkin'],
    'transform_code': ["df['skin_avg'] = df[['rater1', 'rater2']].mean(axis=1)\ndf = df[df['photoID'].notna()]  # ensure photo existed for rating\ndf = df[df['skin_avg'].notna()]\ndf = df[(df['skin_avg'] <= 0.4) | (df['skin_avg'] >= 0.6)].copy()\ndf['DarkSkin'] = (df['skin_avg'] >= 0.6).astype(int)"]}],
  'control_variables': [{'description': 'Number of matches (games) the player and referee encountered each other; used as an exposure variable (offset) in the count model so the model estimates rates of red cards per game.',
    'is_moderator': False,
    'moderator_on': No

## Extract Model Class Used

In [7]:
model_info = format_model_info(multirun_analyses, num_analyses, llm_assistant)
model_info

[2025-12-11 03:23:43.61][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-12-11 03:23:57.24][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.64 seconds
[2025-12-11 03:23:57.25][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-11 03:23:57.27][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-11 03:24:09.43][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  12.17 seconds
[2025-12-11 03:24:09.43][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-11 03:24:09.45][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-11 03:24:20.49][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.04 seconds
[2025-12-11 03:24:20.54][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)

{0: '{\n  "model_library": "statsmodels (statsmodels.formula.api)",\n  "model_class": "OLS (Ordinary Least Squares via statsmodels.formula.api.ols)",\n  "model_parameters": "No explicit hyperparameters set (defaults used). Preprocessing: df[\'FertilityGroup\'] cast to categorical. Model terms specified in formula: categorical encoding and interaction C(FertilityGroup) * InRelationship; controls SureAvg and ReportedCycleLength.",\n  "model_formula_fitting_code": "df[\'FertilityGroup\'] = df[\'FertilityGroup\'].astype(\'category\')\\nformula = \'AvgReligiosity ~ C(FertilityGroup) * InRelationship + SureAvg + ReportedCycleLength\'\\nresults = smf.ols(formula=formula, data=df).fit()"\n}',
 1: '{\n  "model_library": "statsmodels (statsmodels.formula.api)",\n  "model_class": "OLS (Ordinary Least Squares) via statsmodels.formula.api.ols",\n  "model_parameters": "No explicit hyperparameters set; uses default OLS settings. Model specification includes an interaction and categorical coding: AvgR

## Extract Final Answer/Conclusion

Each of the BLADE tasks revolves around a question with the following format:

*What is the effect of [something] on [potential response]?*

It seems that often times the feature to use for the response is not deterministic; the model will have to use some sort of proxy to estimate it. The explanatory features are typically a little bit more clear, but still often require transformations and judgment calls on interpretation and use.

Importantly, this type of question ensures there is a binary answer. While the LLM data scientist does not explicitly spit out a yes/no value, it does write two functions: one which preprocesses the data and another that performs some sort of analysis. Theoretically, we could take the output of the analysis and inspect it to determine whether or not the feature of interest had an effect on the response.

In [11]:
# get path to the dataset
dataset_name = multirun_analyses['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

# load the dataset
data = pd.read_csv(dataset_path)

# create dictionaries to store the imported functions
transform_functions = {}
model_functions = {}

# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    # dynamically import the module
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    # extract transform and model functions
    transform_functions[i] = module.transform
    model_functions[i] = module.model

In [12]:
# Run transform functions on the dataset
transformed_datasets = {}
for i, transform_func in transform_functions.items():
    try:
        transformed_datasets[i] = transform_func(data.copy())  # use copy of dataset
        print(f"[Transform {i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform {i}] Failed with error: {e}")
        transformed_datasets[i] = None

# Run model functions on the transformed datasets
model_results = {}
for i, model_func in model_functions.items():
    try:
        if transformed_datasets[i] is None:
            print(f"[Model {i}] Skipping — transform step failed.")
            continue

        model_results[i] = model_func(transformed_datasets[i].copy())  # use copy
        print(f"[Model {i}] Completed successfully.")
    except Exception as e:
        print(f"[Model {i}] Failed with error: {e}")
        model_results[i] = None


[Transform 0] Completed successfully.
[Transform 1] Completed successfully.
[Transform 2] Completed successfully.
[Model 0] Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1] Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 2] Completed successfully.


In [13]:
# view the first model result as a sanity check
model_results[0]

In [14]:
# create storage object for final answers
final_answer_code = {}

# read task from info.json in the dataset directory
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)
task = info_json['research_questions']

for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model output from object made in previous cell
    model_output = model_results[i]
    
    # call the helper function
    final_answer_code[i] = write_final_answer_code(llm_assistant, task,
                                                   independent_variable,
                                                   dependent_variable,
                                                   model_code, model_output)

[2025-12-05 01:19:43.71][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:20:25.05][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  41.34 seconds
[2025-12-05 01:20:25.06][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:20:25.09][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:20:51.40][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  26.31 seconds
[2025-12-05 01:20:51.40][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:20:51.42][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:21:28.83][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  37.

In [15]:
# run the final answer code
final_answer_code

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts the coefficient, standard error, p-value, confidence interval, and incidence-rate ratio (IRR)\n    for the \'DarkSkin\' predictor from a fitted statsmodels results object (possibly with clustered robust SEs).\n    Returns a dictionary with:\n      - "object": dict of numeric results\n      - "description": plain-language interpretation of the result in the study context\n    """\n    import numpy as np\n\n    res = model_output\n\n    # Helper to safely get an attribute (works if wrapped or plain)\n    def _getattr(obj, attr):\n        return getattr(obj, attr, None)\n\n    # Try to locate parameter names and values\n    params = _getattr(res, "params")\n    # Some wrappers might store results under .results\n    if params is None and hasattr(res, "results"):\n        params = _getattr(res.results, "params")\n\n    if params is None:\n        raise ValueError("Could not locate model parameters (params) in the provided 

In [16]:
# loop through final answer code and dynamically execute the functions
final_answer_functions = {}
for i in range(num_analyses):
    # create a namespace dictionary to execute the code in
    namespace = {}
    
    # compile and execute the code
    compiled_code = compile(final_answer_code[i], f"<final_answer_code_{i}>", "exec")
    exec(compiled_code, namespace)
    
    # extract function from namespace
    final_answer_functions[i] = namespace['extract_final_answer']

# run the final answer functions on the model results
final_answers = [final_answer_functions[i](model_results[i]) for i in range(num_analyses)]

In [17]:
conclusions = {}
for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model interpretation code
    try:
        interpretation_code = final_answer_code[i]
    except (KeyError, IndexError, TypeError):
        interpretation_code = None
    
    try: 
        interpretation_output = final_answers[i]
    except (KeyError, IndexError, TypeError):
        interpretation_output = None
    
    # call the helper function
    conclusions[i] = make_conclusion(llm_assistant, task, independent_variable,
                                     dependent_variable, model_code,
                                     interpretation_code, interpretation_output)

[2025-12-05 01:21:29.13][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:21:35.60][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.47 seconds
[2025-12-05 01:21:35.60][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:21:35.62][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:21:42.31][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.69 seconds
[2025-12-05 01:21:42.32][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:21:42.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:21:50.69][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.36 

In [18]:
conclusions

{0: '{"answer": "Not enough information", "justification": "The estimated IRR = 1.146 (14.6% higher rate) favors dark-skinned players, but the effect is not statistically significant (p = 0.073) and the 95% CI includes 1 (0.987–1.331). Therefore the analysis does not provide strong enough evidence to conclude they are more likely to receive red cards."}',
 1: '{\n  "answer": "Yes",\n  "justification": "The adjusted negative-binomial model estimates an IRR = 1.213 (95% CI 1.036–1.420, p = 0.016) for SkinDark, indicating dark-skinned players receive red cards at a statistically significantly higher rate (~21% higher per game) than light-skinned players (offset for games and controlling for listed covariates)."\n}',
 2: '{\n  "answer": "Yes",\n  "justification": "Both the primary continuous negative-binomial model and the binary robustness model show a statistically significant positive association: SkinTone coef=0.268 (RR=1.31, 95% CI 1.09–1.58, p≈0.005) and IsDark coef=0.212 (RR=1.24, 9

In [19]:
llm_judge = llm(provider=llm_provider, model=llm_model)

[2025-12-05 01:21:50.84][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


In [20]:
data_head = data.head(10)

In [21]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final JSON object.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in JSON format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)

In [22]:
judge_user_prompt = (
    f"Research Question / Context:\n{task}\n\n"
    "Here is a sample of the dataset to understand the structure and variables:\n"
    f"{data_head}\n\n"
    "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
    "==================== TRIAL 0 ====================\n\n"
    "Independent Variables:\n"
    f"{features[0]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[0]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[0]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[0]}\n\n"
    "Conclusion:\n"
    f"{conclusions[0]}\n\n"
    "==================== TRIAL 1 ====================\n\n"
    "Independent Variables:\n"
    f"{features[1]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[1]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[1]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[1]}\n\n"
    "Conclusion:\n"
    f"{conclusions[1]}\n\n"
    "Now, following your reasoning plan, provide similarity ratings as JSON only."
)

In [23]:
final_scores = llm_judge.generate([{"role": "system",
                                        "content": judge_system_prompt},
                                       {"role": "user",
                                        "content": judge_user_prompt}])

[2025-12-05 01:21:51.49][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:22:05.44][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.95 seconds
[2025-12-05 01:22:05.44][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
